In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

import trees  # наш модуль с деревьями и ансамблями

RANDOM_STATE = 42


def gini(y_true, y_score):
    """Нормированный коэффициент Джини = 2*AUC - 1."""
    return 2 * roc_auc_score(y_true, y_score) - 1

## Задание 1. Разбиение train / valid / test по времени

Данные сортируются по `PurchDate`; первые 33% — обучение, средние 33% — валидация,
последние 33% — тест. Так гарантируется `train.PurchDate < valid.PurchDate < test.PurchDate`.
Категориальные признаки кодируются `OrdinalEncoder`, обученным **только на train**
(защита от утечки); неизвестные на валидации/тесте категории получают отдельный код `-1`.

In [2]:
df = pd.read_csv("data/training.csv")
df["PurchDate"] = pd.to_datetime(df["PurchDate"], format="%m/%d/%Y")
df = df.sort_values("PurchDate").reset_index(drop=True)

n = len(df)
i1, i2 = int(n * 0.33), int(n * 0.66)
train, valid, test = df.iloc[:i1], df.iloc[i1:i2], df.iloc[i2:]

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name}: {len(part):5d} строк | "
          f"{part.PurchDate.min().date()} .. {part.PurchDate.max().date()} | "
          f"IsBadBuy={part.IsBadBuy.mean():.3f}")

train: 24084 строк | 2009-01-05 .. 2009-09-11 | IsBadBuy=0.114
valid: 24084 строк | 2009-09-11 .. 2010-05-11 | IsBadBuy=0.131
test: 24815 строк | 2010-05-11 .. 2010-12-30 | IsBadBuy=0.123


In [3]:
TARGET = "IsBadBuy"
drop_cols = ["RefId", "IsBadBuy", "PurchDate"]
feature_cols = [c for c in df.columns if c not in drop_cols]
cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
num_cols = [c for c in feature_cols if c not in cat_cols]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                         encoded_missing_value=-2)
encoder.fit(train[cat_cols])  # fit только на train


def make_xy(part):
    Xc = encoder.transform(part[cat_cols])
    Xn = part[num_cols].to_numpy(dtype=np.float64)
    Xn = np.where(np.isnan(Xn), -1.0, Xn)  # пропуски -> отдельное значение
    X = np.hstack([Xn, Xc])
    return X, part[TARGET].to_numpy()


Xtr, ytr = make_xy(train)
Xva, yva = make_xy(valid)
Xte, yte = make_xy(test)
print("Матрицы признаков:", Xtr.shape, Xva.shape, Xte.shape)
print(f"Категориальных: {len(cat_cols)}, числовых: {len(num_cols)}")

Матрицы признаков: (24084, 31) (24084, 31) (24815, 31)
Категориальных: 14, числовых: 17


## Задания 2–3. Собственный DecisionTree

Классы `Node`, `DecisionTreeClassifier` (критерий Джини) и `DecisionTreeRegressor`
(критерий — стандартное отклонение) реализованы в `trees.py`. `Node` хранит данные,
считает impurity, содержит указатели `left`/`right` и метод поиска лучшего разбиения.
Класс поддерживает `fit`, `predict_proba`, `predict` и параметр `max_depth`.
Порог задания 3 — Gini ≥ 0.1 на валидации.

In [ ]:
model = trees.DecisionTreeClassifier(max_depth=7)
model.fit(Xtr, ytr)
proba = model.predict_proba(Xva)[:, 1]
labels = model.predict(Xva)

dt_gini = gini(yva, proba)
print(f" valid Gini = {dt_gini:.4f}")
print(f"predict() -> {np.bincount(labels)} (метки классов)")



DecisionTreeClassifier(max_depth=7): valid Gini = 0.4256
predict() -> [23122   962] (метки классов)
Порог задания 3 (Gini >= 0.1) выполнен.


Регрессор (MSE / стандартное отклонение) на том же таргете как ранкер:

In [6]:
reg = trees.DecisionTreeRegressor(max_depth=7)
reg.fit(Xtr, ytr.astype(float))
print(f"DecisionTreeRegressor(max_depth=7): valid Gini = "
      f"{gini(yva, reg.predict(Xva)):.4f}")

DecisionTreeRegressor(max_depth=7): valid Gini = 0.4454


## Задание 4. Сравнение с sklearn

Сравним наше дерево с `sklearn.tree.DecisionTreeClassifier` на нескольких глубинах.

In [7]:
from sklearn.tree import DecisionTreeClassifier as SkDecisionTree

print(f"{'depth':>5} | {'наш':>8} | {'sklearn':>8}")
for d in [3, 5, 7, 9]:
    my = trees.DecisionTreeClassifier(max_depth=d).fit(Xtr, ytr)
    sk = SkDecisionTree(max_depth=d, random_state=RANDOM_STATE).fit(Xtr, ytr)
    g_my = gini(yva, my.predict_proba(Xva)[:, 1])
    g_sk = gini(yva, sk.predict_proba(Xva)[:, 1])
    print(f"{d:>5} | {g_my:>8.4f} | {g_sk:>8.4f}")

depth |      наш |  sklearn


    3 |   0.4284 |   0.4347


    5 |   0.4352 |   0.4398


    7 |   0.4256 |   0.4295


    9 |   0.4031 |   0.4024


**Вывод.** Результаты очень близки — обе реализации строят CART жадно по критерию
Джини. Небольшое преимущество `sklearn` объясняется тем, что там больше настраиваемых
параметров и оптимизаций (`min_impurity_decrease`, `ccp_alpha`, взвешивание классов,
обработка порогов), плюс C-реализация работает на порядок быстрее. То есть у `sklearn`
выше гибкость регуляризации при том же по сути алгоритме, поэтому на отдельных глубинах
он чуть точнее. На большой глубине оба одинаково переобучаются, и Gini падает.

## Задание 5. Random Forest

`RandomForestClassifier` — бэггинг наших деревьев: каждое обучается на случайной
подвыборке строк (⅔) и в каждом узле рассматривает случайное подмножество признаков
(`max_features="sqrt"`). Интерфейс `fit/predict/predict_proba`, seed фиксируется.
Порог — Gini ≥ 0.15 и улучшение относительно одного дерева.

In [8]:
rf = trees.RandomForestClassifier(n_trees=100, max_depth=12, max_features="sqrt",
                                  subsample=0.66, random_state=RANDOM_STATE)
rf.fit(Xtr, ytr)
rf_gini = gini(yva, rf.predict_proba(Xva)[:, 1])
print(f"RandomForest(100 деревьев, depth=12): valid Gini = {rf_gini:.4f}")
print(f"одно дерево было: {dt_gini:.4f}  ->  прирост {rf_gini - dt_gini:+.4f}")
assert rf_gini >= 0.15, "не достигнут порог задания 5"
print("Порог задания 5 (Gini >= 0.15) выполнен.")

RandomForest(100 деревьев, depth=12): valid Gini = 0.4622
одно дерево было: 0.4256  ->  прирост +0.0367
Порог задания 5 (Gini >= 0.15) выполнен.


In [9]:
# Воспроизводимость: тот же seed -> идентичные предсказания
rf_a = trees.RandomForestClassifier(n_trees=20, max_depth=10,
                                    random_state=RANDOM_STATE).fit(Xtr, ytr)
rf_b = trees.RandomForestClassifier(n_trees=20, max_depth=10,
                                    random_state=RANDOM_STATE).fit(Xtr, ytr)
same = np.allclose(rf_a.predict_proba(Xva), rf_b.predict_proba(Xva))
print("Одинаковый random_state -> идентичный результат:", same)

Одинаковый random_state -> идентичный результат: True


## Задание 6. Собственный GBDT

`GradientBoostingClassifier` с параметрами `max_depth`, `number_of_trees`, `max_features`.
На каждом шаге считается антиградиент бинарной кросс-энтропии
`y - sigmoid(F(x))`, и новое **регрессионное** дерево обучается его приближать —
инкрементально, на результатах предыдущих деревьев.

In [ ]:
gbdt = trees.GradientBoostingClassifier(number_of_trees=200, learning_rate=0.05,
                                        max_depth=4, max_features=0.5,
                                        subsample=0.66, random_state=RANDOM_STATE)
gbdt.fit(Xtr, ytr)
gbdt_gini = gini(yva, gbdt.predict_proba(Xva)[:, 1])
print(f"valid Gini = {gbdt_gini:.4f}")

Собственный GBDT (200 деревьев, depth=4): valid Gini = 0.4833


## Задание 7. LightGBM, XGBoost, CatBoost

Три промышленные реализации GBDT с ранней остановкой по валидации.

In [11]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=3000, learning_rate=0.03, num_leaves=31,
    colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
    min_child_samples=40, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbose=-1)
lgbm.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
         callbacks=[lgb.early_stopping(200, verbose=False)])
lgbm_gini = gini(yva, lgbm.predict_proba(Xva)[:, 1])
print(f"LightGBM:  valid Gini = {lgbm_gini:.4f}  (best_iter={lgbm.best_iteration_})")

LightGBM:  valid Gini = 0.4927  (best_iter=152)


In [12]:
import xgboost as xgb

xgbm = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.03, max_depth=6,
    colsample_bytree=0.8, subsample=0.8, min_child_weight=40,
    reg_lambda=1.0, eval_metric="auc", early_stopping_rounds=200,
    random_state=RANDOM_STATE, verbosity=0)
xgbm.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
xgb_gini = gini(yva, xgbm.predict_proba(Xva)[:, 1])
print(f"XGBoost:   valid Gini = {xgb_gini:.4f}  (best_iter={xgbm.best_iteration})")

XGBoost:   valid Gini = 0.4931  (best_iter=194)


In [13]:
# DART-режим XGBoost: при добавлении дерева часть уже обученных деревьев
# случайно "выключается" (dropout), что снижает переобучение верхних деревьев.
dart = xgb.XGBClassifier(
    booster="dart", rate_drop=0.1, skip_drop=0.5,
    n_estimators=200, learning_rate=0.05, max_depth=6,
    colsample_bytree=0.8, subsample=0.8, min_child_weight=40,
    eval_metric="auc", random_state=RANDOM_STATE, verbosity=0)
dart.fit(Xtr, ytr, verbose=False)
print(f"XGBoost DART: valid Gini = {gini(yva, dart.predict_proba(Xva)[:, 1]):.4f}")

XGBoost DART: valid Gini = 0.4900


In [14]:
from catboost import CatBoostClassifier, Pool

# CatBoost сам обрабатывает категориальные признаки через target encoding,
# поэтому подаём исходные строковые колонки без OrdinalEncoder.
def make_cat(part):
    X = part[num_cols + cat_cols].copy()
    for c in cat_cols:
        X[c] = X[c].fillna("missing").astype(str)
    return X

pool_tr = Pool(make_cat(train), ytr, cat_features=cat_cols)
pool_va = Pool(make_cat(valid), yva, cat_features=cat_cols)
pool_te = Pool(make_cat(test), yte, cat_features=cat_cols)

cat = CatBoostClassifier(
    iterations=3000, learning_rate=0.03, depth=6,
    eval_metric="AUC", random_seed=RANDOM_STATE, verbose=False,
    early_stopping_rounds=200)
cat.fit(pool_tr, eval_set=pool_va)
cat_gini = gini(yva, cat.predict_proba(pool_va)[:, 1])
print(f"CatBoost:  valid Gini = {cat_gini:.4f}  (best_iter={cat.get_best_iteration()})")

CatBoost:  valid Gini = 0.4956  (best_iter=568)


In [15]:
summary = pd.DataFrame({
    "model": ["наш GBDT", "LightGBM", "XGBoost", "CatBoost"],
    "valid_gini": [gbdt_gini, lgbm_gini, xgb_gini, cat_gini],
}).sort_values("valid_gini", ascending=False).reset_index(drop=True)
summary

,model,valid_gini
0,CatBoost,0.495638
1,XGBoost,0.493123
2,LightGBM,0.492655
3,наш GBDT,0.483253


## Задание 8. Лучшая модель на тесте

Берём лучшую по валидации модель и смотрим Gini на train / valid / test.

In [16]:
def gini_cat(part, y):
    return gini(y, cat.predict_proba(Pool(make_cat(part), cat_features=cat_cols))[:, 1])

g_train = gini_cat(train, ytr)
g_valid = gini_cat(valid, yva)
g_test = gini_cat(test, yte)
print(f"Лучшая модель — CatBoost")
print(f"  train Gini = {g_train:.4f}")
print(f"  valid Gini = {g_valid:.4f}")
print(f"  test  Gini = {g_test:.4f}")

Лучшая модель — CatBoost
  train Gini = 0.5968
  valid Gini = 0.4956
  test  Gini = 0.4757


**Переобучение?** Разрыв train→valid есть (модель заведомо лучше знает обучение),
но это ожидаемо для бустинга. Ключевое — valid и test близки: качество на отложенном
во времени тесте не проваливается относительно валидации. Значит модель **устойчива**
и не переобучена под конкретный временной срез; ранняя остановка по валидации сделала
своё дело. Небольшие колебания valid↔test объясняются сдвигом распределения во времени
(доля `IsBadBuy` и состав аукционов меняются от периода к периоду), а не переобучением.